# Self-Consistency와 표본 합의

같은 질문을 독립적으로 여러 번 실행한 최종 답을 어떻게 모으고, 합의 결과의 한계를 어떻게 기록할까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

`run(messages)`는 이 노트북에서 작성하는 보조 함수입니다. 역할·내용 목록을 받아 `model.invoke()`를 한 번 실행하고, 답 문자열·사용량·경과 초를 담은 딕셔너리를 반환합니다. `def` 셀은 함수를 정의할 뿐이며 뒤에서 `run(...)`을 호출할 때 모델 요청이 시작됩니다. `mention_flags(text)`는 지정 단어가 문자열에 있는지 표시한 딕셔너리를 반환합니다. 답의 뜻을 판단하는 모델 호출은 포함하지 않습니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 반복 조건 고정

모델, 질문, 프롬프트와 평가 기준을 유지한 채 세 번 호출합니다.

In [5]:
# LABEL 첫 줄 형식을 고정한 같은 프롬프트를 세 번 독립 호출합니다.
messages = [
    ("system", "첫 줄에 LABEL: 배송, LABEL: 교환, LABEL: 환불 중 하나만 쓰고 "
               "둘째 줄에 확인 근거를 쓰세요."),
    ("user", TASK),
]
samples = [run(messages) for _ in range(3)]
for index, item in enumerate(samples, 1):
    print(f"표본 {index}:", item["text"], item["usage"], item["latency_s"])

표본 1: LABEL: 교환

확인 근거: 어제 받은 텀블러 뚜껑이 깨졌고, 새 상품으로 바꾸고 싶다고 하셨으며, 교환 접수에 파손 사진이 필요합니다.

필요한 다음 행동: 파손된 뚜껑 사진과 주문번호 A-1042를 함께 보내주시면 교환 접수를 진행할 수 있습니다.

고객에게 보낼 한 문장: 파손 사진과 주문번호 A-1042를 보내주시면 확인 후 새 상품 교환을 바로 도와드리겠습니다. {'input_tokens': 117, 'output_tokens': 133, 'total_tokens': 250, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.46
표본 2: LABEL: 교환
파손된 뚜껑 사진을 먼저 보내주시면 교환 접수를 진행할 수 있습니다.
고객에게는 “뚜껑 파손 사진을 첨부해 주시면 새 상품으로 교환 접수 도와드리겠습니다.”라고 안내하세요. {'input_tokens': 117, 'output_tokens': 67, 'total_tokens': 184, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.007
표본 3: LABEL: 교환
파손 사진을 첨부해 주시면 교환 접수를 진행할 수 있습니다.
고객에게 보낼 한 문장: 파손된 뚜껑 사진을 보내주시면 확인 후 새 상품 교환 절차를 안내드리겠습니다. {'input_tokens': 117, 'output_tokens': 63, 'total_tokens': 180, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.0


## 최종 답에서 분류 추출

공개된 최종 답의 분류 단어만 집계합니다. 내부 추론은 수집하지 않습니다.

`splitlines()`는 응답 문자열을 줄 목록으로 나누고 `[0]`은 첫 줄을 선택합니다. `startswith("LABEL: ")`로 접두어를 검사한 뒤 그 길이 이후의 부분을 슬라이스하여 라벨을 읽습니다. `allowed`는 허용한 라벨의 집합이며 포함되지 않은 값은 형식 오류로 기록합니다.

이 코드의 과반 기준은 유효 응답 수가 아니라 전체 표본 수입니다. `count > len(samples) // 2`이므로 세 번 중 같은 유효 라벨이 두 번 이상 있어야 합니다. 교환·형식 오류·형식 오류라면 한 건만 유효해도 합의 없음입니다. `next(..., "합의 없음")`는 조건을 만족하는 라벨이 없을 때 사용할 값을 지정합니다.

이것은 최종 라벨 합의를 읽는 축소 실습입니다. 연구에서 다루는 여러 추론 경로의 표집 전체를 구현하지 않습니다. 별도 호출이어도 같은 모델의 편향이 반복될 수 있으며 다수결이 사실 확인을 대신하지 않습니다.

In [6]:
# LABEL 접두어와 열거형을 정확히 검사하고 전체 표본의 엄격 과반수만 합의로 인정합니다.
allowed = {"배송", "교환", "환불"}
labels = []
for item in samples:
    first_line = item["text"].splitlines()[0].strip()
    has_prefix = first_line.startswith("LABEL: ")
    label = first_line[len("LABEL: "):].strip() if has_prefix else "형식 오류"
    labels.append(label if label in allowed else "형식 오류")
valid = [label for label in labels if label in allowed]
counts = {label: valid.count(label) for label in sorted(set(valid))}
consensus = next((label for label, count in counts.items()
                  if count > len(samples) // 2), "합의 없음")
print("표본 분류:", labels, "빈도:", counts, "합의:", consensus)

표본 분류: ['교환', '교환', '교환'] 빈도: {'교환': 3} 합의: 교환


## 품질과 비용 함께 확인

합의 여부와 별개로 각 원문이 고정 루브릭을 충족했는지 확인합니다.

In [7]:
# 사용량이 하나라도 없으면 0으로 바꾸지 않고 전체 토큰을 None으로 남깁니다.
token_values = [item["usage"].get("total_tokens") if item["usage"] else None
                for item in samples]
summary = {
    "sample_count": len(samples),
    "unanimous": len(set(labels)) == 1 and "형식 오류" not in labels,
    "mention_flags": [mention_flags(item["text"]) for item in samples],
    "total_tokens": sum(token_values) if all(v is not None for v in token_values) else None,
    "total_latency_s": round(sum(item["latency_s"] for item in samples), 3),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "sample_count": 3,
  "unanimous": true,
  "mention_flags": [
    {
      "교환": true,
      "A-1042": true,
      "사진": true
    },
    {
      "교환": true,
      "A-1042": false,
      "사진": true
    },
    {
      "교환": true,
      "A-1042": false,
      "사진": true
    }
  ],
  "total_tokens": 614,
  "total_latency_s": 3.467
}
